# E-Commerce Customer Segmentation using K-Means and DBSCAN
**Course:** MLE - AI201B | **CA2 – Case Study 3**

## Objective
Segment online retail customers using **RFM (Recency, Frequency, Monetary)** features. Compare K-Means and DBSCAN, reduce dimensions using PCA, visualize customer groups with t-SNE, and suggest marketing actions for each segment.

## Dataset
The notebook downloads the UCI Online Retail dataset. It contains transaction-level records from an online retailer.

In [ ]:
# Install any missing package (Colab usually already has these)
!pip -q install openpyxl

In [ ]:
import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

sns.set_theme()

## 1. Load the dataset
We download the public UCI dataset. A transaction has an invoice number, product, quantity, price, customer ID and invoice date.

In [ ]:
url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
zip_path = "online_retail.zip"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as z:
    excel_name = [n for n in z.namelist() if n.lower().endswith(".xlsx")][0]
    z.extract(excel_name, ".")

df = pd.read_excel(excel_name)
print("Dataset shape:", df.shape)
display(df.head())

## 2. Data cleaning
Remove rows without a customer ID, cancelled invoices, and invalid/non-positive quantity or price. Create transaction value as Quantity × UnitPrice.

In [ ]:
df = df.dropna(subset=["CustomerID"]).copy()
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df["CustomerID"] = df["CustomerID"].astype(int).astype(str)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

print("Cleaned dataset shape:", df.shape)
display(df.head())

## 3. Build RFM features
- **Recency:** number of days since the customer's most recent purchase (lower is better).
- **Frequency:** number of distinct invoices/orders (higher means more frequent purchases).
- **Monetary:** total money spent (higher means greater customer value).

In [ ]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum")
).reset_index()

display(rfm.head())
print("Number of customers:", len(rfm))
display(rfm[["Recency", "Frequency", "Monetary"]].describe())

## 4. Explore and prepare the RFM data
RFM values are highly skewed, especially Monetary. Apply log transformation and standardization so that large numerical scales do not dominate clustering.

In [ ]:
features = rfm[["Recency", "Frequency", "Monetary"]].copy()
features_log = np.log1p(features)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_log)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, features.columns):
    sns.histplot(rfm[col], bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 5. K-Means clustering
K-Means assigns each customer to one of K clusters. We compare several K values using inertia (elbow method) and silhouette score, then fit the final model.

In [ ]:
inertias, silhouettes = [], []
k_values = range(2, 8)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(k_values), silhouettes, marker="o")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Score")
plt.tight_layout()
plt.show()

best_k = list(k_values)[int(np.argmax(silhouettes))]
print("K selected by highest silhouette score:", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm["KMeans_Cluster"] = kmeans.fit_predict(X_scaled)
print("K-Means silhouette score:", round(silhouette_score(X_scaled, rfm["KMeans_Cluster"]), 3))
display(rfm["KMeans_Cluster"].value_counts().sort_index())

## 6. DBSCAN clustering
DBSCAN is a density-based algorithm. It does not require a predefined number of clusters and can label unusual points as noise (`-1`). Its result depends on `eps` and `min_samples`; the values below are starting parameters and can be adjusted.

In [ ]:
dbscan = DBSCAN(eps=0.7, min_samples=10)
rfm["DBSCAN_Cluster"] = dbscan.fit_predict(X_scaled)

print("DBSCAN cluster counts (label -1 means noise):")
display(rfm["DBSCAN_Cluster"].value_counts().sort_index())

db_labels = rfm["DBSCAN_Cluster"].values
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
non_noise = db_labels != -1
if n_db_clusters >= 2 and non_noise.sum() > n_db_clusters:
    print("DBSCAN silhouette score (excluding noise):",
          round(silhouette_score(X_scaled[non_noise], db_labels[non_noise]), 3))
else:
    print("A silhouette score is not meaningful here because DBSCAN found fewer than 2 usable clusters.")

## 7. PCA dimensionality reduction
PCA converts the three standardized RFM features into principal components. The first two components allow a 2D visualization while retaining as much variance as possible.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print("Explained variance by PC1 and PC2:", np.round(pca.explained_variance_ratio_, 3))
print("Total variance retained:", round(pca.explained_variance_ratio_.sum(), 3))

pca_plot = pd.DataFrame({"PC1": X_pca[:, 0], "PC2": X_pca[:, 1],
                         "KMeans_Cluster": rfm["KMeans_Cluster"].astype(str)})
plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_plot, x="PC1", y="PC2", hue="KMeans_Cluster",
                palette="tab10", s=18, alpha=0.7)
plt.title("K-Means customer segments (PCA view)")
plt.show()

## 8. t-SNE visualization
t-SNE is a nonlinear technique that tries to preserve local neighbourhoods. It is used here for visualization, not as an input to the clustering algorithms. We sample customers to keep execution time manageable.

In [ ]:
sample_size = min(3000, len(X_scaled))
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_idx]
labels_sample = rfm["KMeans_Cluster"].to_numpy()[sample_idx]

perplexity = min(30, max(5, (sample_size - 1) // 3))
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42,
            init="pca", learning_rate="auto")
X_tsne = tsne.fit_transform(X_sample)

tsne_plot = pd.DataFrame({"tSNE-1": X_tsne[:, 0], "tSNE-2": X_tsne[:, 1],
                          "Cluster": labels_sample.astype(str)})
plt.figure(figsize=(8, 6))
sns.scatterplot(data=tsne_plot, x="tSNE-1", y="tSNE-2", hue="Cluster",
                palette="tab10", s=20, alpha=0.7)
plt.title("K-Means customer segments (t-SNE view)")
plt.show()

## 9. Interpret clusters and create marketing personas
Use the original RFM values (not transformed values) to understand each cluster. The persona descriptions below are generated from each cluster's relative RFM profile and should be checked against the displayed table.

In [ ]:
cluster_profile = rfm.groupby("KMeans_Cluster").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean")
).round(2)

display(cluster_profile)

def persona(row):
    # Relative profile-based rules; thresholds are dataset-dependent.
    if row["Avg_Recency"] <= rfm["Recency"].median() and row["Avg_Frequency"] >= rfm["Frequency"].median() and row["Avg_Monetary"] >= rfm["Monetary"].median():
        return "Champions / High-value loyal customers"
    if row["Avg_Recency"] > rfm["Recency"].median():
        return "At-risk / Need re-engagement"
    if row["Avg_Frequency"] <= rfm["Frequency"].median() and row["Avg_Monetary"] <= rfm["Monetary"].median():
        return "Low-engagement / New or occasional customers"
    return "Potential loyalists / Growth opportunity"

cluster_profile["Marketing_Persona"] = cluster_profile.apply(persona, axis=1)
display(cluster_profile)

### Suggested marketing actions
| Persona | Suggested action |
|---|---|
| Champions / High-value loyal customers | VIP benefits, early access, loyalty rewards, referral offers |
| At-risk / Need re-engagement | Personalized reminders, win-back discounts, feedback requests |
| Low-engagement / New or occasional customers | Welcome journeys, product recommendations, limited-time first-repeat-purchase offers |
| Potential loyalists / Growth opportunity | Bundles, cross-selling, loyalty points, relevant recommendations |

The table is a practical guide. Use the cluster profile values to validate which action fits each segment.

## 10. Compare K-Means and DBSCAN
- **K-Means:** produces a fixed number of clusters, works well for compact groups, and requires choosing K.
- **DBSCAN:** discovers dense regions, can identify noise/outliers, and does not require K; however, it is sensitive to `eps` and may struggle when clusters have different densities.
- Compare the cluster counts and silhouette scores above. A higher silhouette score generally indicates better separation, but it should not be the only measure used.

In [ ]:
comparison = pd.DataFrame({
    "Algorithm": ["K-Means", "DBSCAN"],
    "Clusters_found": [
        rfm["KMeans_Cluster"].nunique(),
        n_db_clusters
    ],
    "Noise_points": [
        0,
        int((rfm["DBSCAN_Cluster"] == -1).sum())
    ]
})
display(comparison)

## 11. Conclusion
RFM analysis converts transaction history into customer-level behavioural features. K-Means provides actionable customer groups, while DBSCAN offers a density-based comparison and identifies unusual customers. PCA and t-SNE help visualize the segments in two dimensions. Businesses can use the resulting personas to personalize campaigns, improve retention, and allocate marketing budgets more effectively.

## Limitations
1. The dataset is historical and may not represent current customer behaviour.
2. RFM uses only purchase timing, order frequency and spending; it does not include customer demographics or product preferences.
3. Clustering is exploratory: segment labels and marketing actions require business validation.
4. DBSCAN parameters affect the number of clusters and noise points.